In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vimalpandey6388/dkt-assessment-original/skill_builder_data.csv


In [3]:

# ============================================================
# CELL 1: ENVIRONMENT SETUP & DATASET DOWNLOAD
# ============================================================

import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)

# -------------------- Reproducibility --------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------- Device Configuration --------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 55)
print("DKT PIPELINE - ENVIRONMENT CONFIGURATION")
print("=" * 55)

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Using CPU.")

# -------------------- Dataset Configuration --------------------

DATASET_NAME = "vimalpandey6388/dkt-assessment-original"
CSV_FILENAME = "skill_builder_data.csv"

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

# First check whether the dataset is already attached to Kaggle.
csv_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if CSV_FILENAME in files:
        csv_path = os.path.join(root, CSV_FILENAME)
        break

# Download using KaggleHub if the dataset is not attached.
if csv_path is None:
    import kagglehub

    print("\nDataset not found in Kaggle Input.")
    print("Downloading dataset using KaggleHub...")

    dataset_path = kagglehub.dataset_download(DATASET_NAME)

    for root, dirs, files in os.walk(dataset_path):
        if CSV_FILENAME in files:
            csv_path = os.path.join(root, CSV_FILENAME)
            break

if csv_path is None:
    raise FileNotFoundError(
        f"{CSV_FILENAME} was not found. "
        "Please check your Kaggle dataset attachment."
    )

print("\nDataset successfully located!")
print("Dataset path:", csv_path)
print("=" * 55)


DKT PIPELINE - ENVIRONMENT CONFIGURATION
Python version: 3.13.16
PyTorch version: 2.11.0+cu130
NumPy version: 2.1.3
Pandas version: 2.3.3
Device: cuda
GPU: Tesla T4

Dataset successfully located!
Dataset path: /kaggle/input/datasets/vimalpandey6388/dkt-assessment-original/skill_builder_data.csv


In [4]:

# ============================================================
# CELL 2: LOAD DATASET & INITIAL EXPLORATORY DATA ANALYSIS
# ============================================================

# Load dataset
try:
    df = pd.read_csv(csv_path, low_memory=False)
    print("Dataset loaded successfully using default encoding.")

except UnicodeDecodeError:
    df = pd.read_csv(
        csv_path,
        encoding="latin-1",
        low_memory=False
    )
    print("Dataset loaded successfully using latin-1 encoding.")


# -------------------- Dataset Overview --------------------

print("\n" + "=" * 65)
print("DATASET OVERVIEW")
print("=" * 65)

print(f"Total rows       : {df.shape[0]:,}")
print(f"Total columns    : {df.shape[1]:,}")
print(f"Unique students  : {df['user_id'].nunique():,}")
print(f"Unique skills    : {df['skill_id'].nunique():,}")

print("\nColumn Names:")
print(df.columns.tolist())


# -------------------- Data Types --------------------

print("\n" + "=" * 65)
print("COLUMN DATA TYPES")
print("=" * 65)

print(df.dtypes)


# -------------------- Missing Values --------------------

print("\n" + "=" * 65)
print("MISSING VALUE ANALYSIS")
print("=" * 65)

missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

missing = missing.sort_values(
    by="Missing Percentage",
    ascending=False
)

print(missing[missing["Missing Count"] > 0])


# -------------------- Duplicate Analysis --------------------

print("\n" + "=" * 65)
print("DUPLICATE ANALYSIS")
print("=" * 65)

print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated order_id rows:",
      df.duplicated(subset=["order_id"]).sum())


# -------------------- Sample Records --------------------

print("\n" + "=" * 65)
print("FIRST 5 RECORDS")
print("=" * 65)

display(df.head())

print("\nDataset inspection completed!")


Dataset loaded successfully using latin-1 encoding.

DATASET OVERVIEW
Total rows       : 525,534
Total columns    : 30
Unique students  : 4,217
Unique skills    : 123

Column Names:
['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'bottom_hint', 'opportunity', 'opportunity_original']

COLUMN DATA TYPES
order_id                  int64
assignment_id             int64
user_id                   int64
assistment_id             int64
problem_id                int64
original                  int64
correct                   int64
attempt_count             int64
ms_first_response         int64
tutor_mode               object
answer_type              object


,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,33022537,277618,64525,33139,51424,1,1,1,32454,tutor,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,33022709,277618,64525,33150,51435,1,1,1,4922,tutor,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,35450204,220674,70363,33159,51444,1,0,2,25390,tutor,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,35450295,220674,70363,33110,51395,1,1,1,4859,tutor,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,35450311,220674,70363,33196,51481,1,0,14,19813,tutor,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0



Dataset inspection completed!


In [5]:

# ============================================================
# CELL 3: DATA PREPROCESSING
# ============================================================

data = df.copy()

# Helper function to track preprocessing progress
def show_preprocessing(step, data):
    print(
        f"{step:<48} "
        f"Rows: {len(data):>9,} | "
        f"Students: {data['user_id'].nunique():>6,} | "
        f"Skills: {data['skill_id'].nunique():>4,}"
    )

print("=" * 85)
print("DATA PREPROCESSING PIPELINE")
print("=" * 85)

show_preprocessing("0. Original dataset", data)


# ------------------------------------------------------------
# STEP 1: Remove interactions without skill information
# ------------------------------------------------------------

data = data.dropna(subset=["skill_id"]).copy()

show_preprocessing("1. Remove missing skill_id", data)


# ------------------------------------------------------------
# STEP 2: Keep only original questions
# ------------------------------------------------------------

data = data[data["original"] == 1].copy()

show_preprocessing("2. Keep original questions", data)


# ------------------------------------------------------------
# STEP 3: Keep valid binary responses
# ------------------------------------------------------------

data = data[data["correct"].isin([0, 1])].copy()

show_preprocessing("3. Keep correct responses (0 or 1)", data)


# ------------------------------------------------------------
# STEP 4: Remove duplicate student-skill interactions
# ------------------------------------------------------------

data = data.drop_duplicates(
    subset=["order_id", "skill_id"]
).copy()

show_preprocessing("4. Remove duplicate (order_id, skill_id)", data)


# ------------------------------------------------------------
# STEP 5: Sort interactions chronologically
# ------------------------------------------------------------

data = data.sort_values(
    by=["user_id", "order_id"],
    kind="stable"
).reset_index(drop=True)

show_preprocessing("5. Sort interactions by student and time", data)


# ------------------------------------------------------------
# STEP 6: Keep students with at least 2 interactions
# ------------------------------------------------------------

interaction_counts = data.groupby("user_id")["order_id"].transform("size")

data = data[
    interaction_counts >= 2
].reset_index(drop=True)

show_preprocessing("6. Keep students with >= 2 interactions", data)


# ------------------------------------------------------------
# FINAL PREPROCESSING SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("FINAL PREPROCESSED DATASET")
print("=" * 85)

print(f"Total interactions : {len(data):,}")
print(f"Unique students    : {data['user_id'].nunique():,}")
print(f"Unique skills      : {data['skill_id'].nunique():,}")

print(
    f"Overall correct rate: "
    f"{data['correct'].mean() * 100:.2f}%"
)

print("\nInteractions per student:")

print(
    data.groupby("user_id")
    .size()
    .describe()
    .round(2)
)

print("\nResponse distribution:")

print(data["correct"].value_counts().sort_index())

print("\nPreprocessing completed successfully!")


DATA PREPROCESSING PIPELINE
0. Original dataset                              Rows:   525,534 | Students:  4,217 | Skills:  123
1. Remove missing skill_id                       Rows:   459,208 | Students:  4,163 | Skills:  123
2. Keep original questions                       Rows:   433,161 | Students:  4,163 | Skills:  123
3. Keep correct responses (0 or 1)               Rows:   433,161 | Students:  4,163 | Skills:  123
4. Remove duplicate (order_id, skill_id)         Rows:   312,232 | Students:  4,163 | Skills:  123
5. Sort interactions by student and time         Rows:   312,232 | Students:  4,163 | Skills:  123
6. Keep students with >= 2 interactions          Rows:   312,111 | Students:  4,042 | Skills:  123

FINAL PREPROCESSED DATASET
Total interactions : 312,111
Unique students    : 4,042
Unique skills      : 123
Overall correct rate: 65.43%

Interactions per student:
count    4042.00
mean       77.22
std       147.14
min         2.00
25%        10.00
50%        23.00
75%        6

In [6]:

# ============================================================
# CELL 4: SKILL ENCODING AND DATASET EXPORT
# ============================================================

# 1. Create a continuous index for every skill
skill_ids = sorted(data["skill_id"].unique())

skill_to_idx = {
    skill_id: idx
    for idx, skill_id in enumerate(skill_ids)
}

data["skill_idx"] = data["skill_id"].map(skill_to_idx).astype(int)

# 2. Create a mapping between skill IDs and skill names
skill_names = (
    df.dropna(subset=["skill_id", "skill_name"])
      .groupby("skill_id")["skill_name"]
      .first()
)

skill_map = pd.DataFrame({
    "skill_idx": range(len(skill_ids)),
    "skill_id": skill_ids
})

skill_map["skill_name"] = skill_map["skill_id"].map(skill_names)

# Use a fallback name if the original skill name is missing
skill_map["skill_name"] = skill_map["skill_name"].fillna(
    skill_map["skill_id"].apply(lambda x: f"skill_{x}")
)

# 3. Keep only the columns required by DKT
clean = data[
    ["user_id", "order_id", "skill_idx", "correct"]
].copy()

# 4. Save processed datasets
clean.to_csv("/kaggle/working/clean_interactions.csv", index=False)
skill_map.to_csv("/kaggle/working/skill_map.csv", index=False)

# 5. Display results
print("=" * 75)
print("SKILL ENCODING COMPLETED")
print("=" * 75)

print(f"Total interactions : {len(clean):,}")
print(f"Unique students    : {clean['user_id'].nunique():,}")
print(f"Total skills       : {len(skill_map):,}")
print(f"Skill index range  : 0 to {len(skill_map) - 1}")

print("\nSample interaction data:")
display(clean.head())

print("\nSample skill mapping:")
display(skill_map.head(10))

print("\nSaved files:")
print("/kaggle/working/clean_interactions.csv")
print("/kaggle/working/skill_map.csv")


SKILL ENCODING COMPLETED
Total interactions : 312,111
Unique students    : 4,042
Total skills       : 123
Skill index range  : 0 to 122

Sample interaction data:


,user_id,order_id,skill_idx,correct
0,14,21617623,1,0
1,14,21617623,24,0
2,14,21617623,44,0
3,14,21617632,1,1
4,14,21617632,24,1



Sample skill mapping:


,skill_idx,skill_id,skill_name
0,0,1.0,Box and Whisker
1,1,2.0,Circle Graph
2,2,4.0,Histogram as Table or Graph
3,3,5.0,Number Line
4,4,8.0,Scatter Plot
5,5,9.0,Stem and Leaf Plot
6,6,10.0,Table
7,7,11.0,Venn Diagram
8,8,12.0,Mean
9,9,13.0,Median



Saved files:
/kaggle/working/clean_interactions.csv
/kaggle/working/skill_map.csv


In [7]:

# ============================================================
# CELL 5: STUDENT-LEVEL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# Get all unique student IDs
student_ids = clean["user_id"].unique()

# First split: 80% development pool, 20% final test
train_val_ids, test_ids = train_test_split(
    student_ids,
    test_size=0.20,
    random_state=SEED
)

# Second split: 90% training, 10% validation
train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.10,
    random_state=SEED
)

# Convert to sets for leakage checks
train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

# Verify that no student appears in multiple sets
assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

# Filter interactions using student IDs
train_data = clean[clean["user_id"].isin(train_set)].copy()
val_data = clean[clean["user_id"].isin(val_set)].copy()
test_data = clean[clean["user_id"].isin(test_set)].copy()

# Display split statistics
print("=" * 75)
print("STUDENT-LEVEL DATA SPLIT")
print("=" * 75)

print(f"{'Dataset':<15} {'Students':>12} {'Interactions':>18}")
print("-" * 50)

print(f"{'Training':<15} {len(train_set):>12,} {len(train_data):>18,}")
print(f"{'Validation':<15} {len(val_set):>12,} {len(val_data):>18,}")
print(f"{'Test':<15} {len(test_set):>12,} {len(test_data):>18,}")

print("-" * 50)
print(f"{'Total':<15} {len(student_ids):>12,} {len(clean):>18,}")

print("\nStudent distribution:")
print(f"Training   : {len(train_set)/len(student_ids)*100:.2f}%")
print(f"Validation : {len(val_set)/len(student_ids)*100:.2f}%")
print(f"Test       : {len(test_set)/len(student_ids)*100:.2f}%")

print("\nLeakage verification:")
print("Train ∩ Validation =", len(train_set & val_set))
print("Train ∩ Test       =", len(train_set & test_set))
print("Validation ∩ Test  =", len(val_set & test_set))

print("\nAll split checks passed!")


STUDENT-LEVEL DATA SPLIT
Dataset             Students       Interactions
--------------------------------------------------
Training               2,909            223,085
Validation               324             25,700
Test                     809             63,326
--------------------------------------------------
Total                  4,042            312,111

Student distribution:
Training   : 71.97%
Validation : 8.02%
Test       : 20.01%

Leakage verification:
Train ∩ Validation = 0
Train ∩ Test       = 0
Validation ∩ Test  = 0

All split checks passed!


In [8]:

# ============================================================
# CELL 6: CREATE STUDENT SEQUENCES AND CHUNKS
# ============================================================

MAX_LEN = 200

def make_chunks(student_data, max_len=MAX_LEN):
    chunks = []

    # Process one student at a time
    for user_id, group in student_data.groupby("user_id", sort=False):

        # Sort interactions chronologically
        group = group.sort_values("order_id", kind="stable")

        # Each interaction is represented as (skill_idx, correct)
        sequence = list(zip(
            group["skill_idx"].astype(int),
            group["correct"].astype(int)
        ))

        # Divide long histories into non-overlapping chunks
        for start in range(0, len(sequence), max_len):

            chunk = sequence[start:start + max_len]

            # Need at least two interactions to predict the next response
            if len(chunk) >= 2:
                chunks.append(chunk)

    return chunks


# Create chunks independently for each dataset
train_chunks = make_chunks(train_data)
val_chunks = make_chunks(val_data)
test_chunks = make_chunks(test_data)


# Display sequence statistics
def show_chunk_stats(name, chunks):

    lengths = [len(chunk) for chunk in chunks]

    print(f"\n{name}")
    print("-" * 55)
    print(f"Number of chunks       : {len(chunks):,}")
    print(f"Total interactions      : {sum(lengths):,}")
    print(f"Average chunk length    : {np.mean(lengths):.2f}")
    print(f"Minimum chunk length    : {min(lengths)}")
    print(f"Maximum chunk length    : {max(lengths)}")


print("=" * 75)
print("STUDENT SEQUENCE PREPARATION")
print("=" * 75)

show_chunk_stats("TRAINING", train_chunks)
show_chunk_stats("VALIDATION", val_chunks)
show_chunk_stats("TEST", test_chunks)

print("\nExample student sequence:")
print(train_chunks[0][:10])

print("\nSequence preparation completed!")


STUDENT SEQUENCE PREPARATION

TRAINING
-------------------------------------------------------
Number of chunks       : 3,417
Total interactions      : 223,084
Average chunk length    : 65.29
Minimum chunk length    : 2
Maximum chunk length    : 200

VALIDATION
-------------------------------------------------------
Number of chunks       : 382
Total interactions      : 25,700
Average chunk length    : 67.28
Minimum chunk length    : 2
Maximum chunk length    : 200

TEST
-------------------------------------------------------
Number of chunks       : 957
Total interactions      : 63,326
Average chunk length    : 66.17
Minimum chunk length    : 2
Maximum chunk length    : 200

Example student sequence:
[(1, 0), (24, 0), (44, 0), (1, 1), (24, 1), (44, 1), (1, 0), (24, 0), (44, 0), (1, 0)]

Sequence preparation completed!


In [9]:

# ============================================================
# CELL 7: PYTORCH DATASET, COLLATE FUNCTION AND DATALOADERS
# ============================================================

N_SKILLS = len(skill_map)

# Padding token: one index after all valid skill-response combinations
PAD_TOKEN = 2 * N_SKILLS

BATCH_SIZE = 64


class DKT_Dataset(Dataset):

    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


def collate_fn(batch):

    x_batch = []
    next_skill_batch = []
    target_batch = []

    for sequence in batch:

        # Separate skill indices and correctness values
        skills = torch.tensor(
            [item[0] for item in sequence],
            dtype=torch.long
        )

        responses = torch.tensor(
            [item[1] for item in sequence],
            dtype=torch.long
        )

        # Encode skill-response pairs
        encoded = skills + N_SKILLS * responses

        # Current interactions become model inputs
        x_batch.append(encoded[:-1])

        # Next skill is used to select the corresponding model output
        next_skill_batch.append(skills[1:])

        # Next response becomes the prediction target
        target_batch.append(responses[1:])

    # Pad sequences within the batch
    x_batch = pad_sequence(
        x_batch,
        batch_first=True,
        padding_value=PAD_TOKEN
    )

    next_skill_batch = pad_sequence(
        next_skill_batch,
        batch_first=True,
        padding_value=0
    )

    target_batch = pad_sequence(
        target_batch,
        batch_first=True,
        padding_value=0
    )

    # True for actual positions, False for padding
    mask = x_batch != PAD_TOKEN

    return x_batch, next_skill_batch, target_batch, mask


# Create datasets
train_dataset = DKT_Dataset(train_chunks)
val_dataset = DKT_Dataset(val_chunks)
test_dataset = DKT_Dataset(test_chunks)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


# Test one batch before training
x, next_skill, y, mask = next(iter(train_loader))

print("=" * 75)
print("PYTORCH DATALOADER VERIFICATION")
print("=" * 75)

print("Number of skills       :", N_SKILLS)
print("Padding token          :", PAD_TOKEN)
print("Batch size             :", BATCH_SIZE)

print("\nBatch tensor shapes:")
print("Input x                :", x.shape)
print("Next skill             :", next_skill.shape)
print("Target y               :", y.shape)
print("Mask                   :", mask.shape)

print("\nInput data type        :", x.dtype)
print("Target data type       :", y.dtype)
print("Mask data type         :", mask.dtype)

print("\nValid prediction positions:", mask.sum().item())
print("Padded positions         :", (~mask).sum().item())

print("\nFirst sequence input:")
print(x[0][:10].tolist())

print("\nFirst sequence next skills:")
print(next_skill[0][:10].tolist())

print("\nFirst sequence targets:")
print(y[0][:10].tolist())

print("\nDataLoader preparation completed!")


PYTORCH DATALOADER VERIFICATION
Number of skills       : 123
Padding token          : 246
Batch size             : 64

Batch tensor shapes:
Input x                : torch.Size([64, 199])
Next skill             : torch.Size([64, 199])
Target y               : torch.Size([64, 199])
Mask                   : torch.Size([64, 199])

Input data type        : torch.int64
Target data type       : torch.int64
Mask data type         : torch.bool

Valid prediction positions: 3121
Padded positions         : 9615

First sequence input:
[51, 51, 51, 174, 98, 98, 221, 98, 221, 221]

First sequence next skills:
[51, 51, 51, 98, 98, 98, 98, 98, 98, 98]

First sequence targets:
[0, 0, 1, 0, 0, 1, 0, 1, 1, 0]

DataLoader preparation completed!


In [10]:

# ============================================================
# CELL 8: ORIGINAL DKT MODEL ARCHITECTURE
# ============================================================

class DKT(nn.Module):

    def __init__(self, n, emb=64, hid=128):

        super().__init__()

        # Convert skill-response indices into dense vectors
        self.emb = nn.Embedding(
            2 * n + 1,
            emb,
            padding_idx=2 * n
        )

        # LSTM learns patterns from student interaction sequences
        self.lstm = nn.LSTM(
            emb,
            hid,
            batch_first=True
        )

        # Predict scores for all skills
        self.out = nn.Linear(hid, n)

    def forward(self, x):

        # Step 1: Embedding
        embedded = self.emb(x)

        # Step 2: LSTM
        hidden, _ = self.lstm(embedded)

        # Step 3: Output layer
        logits = self.out(hidden)

        return logits


# Initialize model
model = DKT(
    n=N_SKILLS,
    emb=64,
    hid=128
).to(DEVICE)


# Test model using one batch
model.eval()

with torch.no_grad():

    sample_logits = model(x.to(DEVICE))


# Display architecture information
print("=" * 75)
print("DKT MODEL VERIFICATION")
print("=" * 75)

print(model)

print("\nModel configuration:")
print("Number of skills       :", N_SKILLS)
print("Embedding dimension    :", 64)
print("LSTM hidden dimension  :", 128)
print("Device                 :", DEVICE)

print("\nInput shape:")
print(x.shape)

print("Output logits shape:")
print(sample_logits.shape)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTotal parameters       :", f"{total_params:,}")
print("Trainable parameters   :", f"{trainable_params:,}")

assert sample_logits.shape == (
    x.shape[0],
    x.shape[1],
    N_SKILLS
)

print("\nDKT model created successfully!")


DKT MODEL VERIFICATION
DKT(
  (emb): Embedding(247, 64, padding_idx=246)
  (lstm): LSTM(64, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=123, bias=True)
)

Model configuration:
Number of skills       : 123
Embedding dimension    : 64
LSTM hidden dimension  : 128
Device                 : cuda

Input shape:
torch.Size([64, 199])
Output logits shape:
torch.Size([64, 199, 123])

Total parameters       : 131,003
Trainable parameters   : 131,003

DKT model created successfully!


In [11]:

# ============================================================
# CELL 9: DKT TRAINING AND VALIDATION
# ============================================================

from sklearn.metrics import roc_auc_score

# Optimizer and loss function
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

loss_fn = nn.BCEWithLogitsLoss()


# Select the prediction corresponding to the next skill
def step_logits(x, next_skill):

    logits = model(x)

    selected_logits = logits.gather(
        dim=2,
        index=next_skill.unsqueeze(-1)
    ).squeeze(-1)

    return selected_logits


# Evaluate model using validation data
def evaluate():

    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for x, next_skill, y, mask in val_loader:

            x = x.to(DEVICE)
            next_skill = next_skill.to(DEVICE)
            y = y.to(DEVICE).float()
            mask = mask.to(DEVICE)

            logits = step_logits(x, next_skill)

            probabilities = torch.sigmoid(logits)

            all_probs.append(probabilities[mask].cpu().numpy())
            all_labels.append(y[mask].cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    auc = roc_auc_score(all_labels, all_probs)

    return auc


# ============================================================
# TRAINING LOOP
# ============================================================

EPOCHS = 20
best_auc = -1.0

print("=" * 75)
print("DKT MODEL TRAINING")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):

    model.train()

    total_loss = 0.0
    batch_count = 0

    for x, next_skill, y, mask in train_loader:

        # Move batch to GPU or CPU
        x = x.to(DEVICE)
        next_skill = next_skill.to(DEVICE)
        y = y.to(DEVICE).float()
        mask = mask.to(DEVICE)

        # Forward pass
        logits = step_logits(x, next_skill)

        # Calculate loss only on valid positions
        loss = loss_fn(
            logits[mask],
            y[mask]
        )

        # Backpropagation
        optimizer.zero_grad()

        loss.backward()

        # Prevent excessively large gradients
        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        # Update model parameters
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

    # Average training loss
    avg_train_loss = total_loss / batch_count

    # Validation AUC
    val_auc = evaluate()

    # Save the best model
    saved_message = ""

    if val_auc > best_auc:

        best_auc = val_auc

        torch.save(
            {
                "state_dict": model.state_dict(),
                "n_skills": N_SKILLS,
                "emb": 64,
                "hid": 128
            },
            "/kaggle/working/dkt_model.pt"
        )

        saved_message = " <-- Best model saved"

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Validation AUC: {val_auc:.4f}"
        f"{saved_message}"
    )


print("\n" + "=" * 75)
print("TRAINING COMPLETED")
print("=" * 75)

print(f"Best Validation AUC: {best_auc:.4f}")
print("Best model saved at: /kaggle/working/dkt_model.pt")


DKT MODEL TRAINING
Epoch 01/20 | Train Loss: 0.5966 | Validation AUC: 0.7868 <-- Best model saved
Epoch 02/20 | Train Loss: 0.4969 | Validation AUC: 0.8208 <-- Best model saved
Epoch 03/20 | Train Loss: 0.4696 | Validation AUC: 0.8272 <-- Best model saved
Epoch 04/20 | Train Loss: 0.4583 | Validation AUC: 0.8314 <-- Best model saved
Epoch 05/20 | Train Loss: 0.4522 | Validation AUC: 0.8339 <-- Best model saved
Epoch 06/20 | Train Loss: 0.4485 | Validation AUC: 0.8353 <-- Best model saved
Epoch 07/20 | Train Loss: 0.4434 | Validation AUC: 0.8366 <-- Best model saved
Epoch 08/20 | Train Loss: 0.4395 | Validation AUC: 0.8371 <-- Best model saved
Epoch 09/20 | Train Loss: 0.4360 | Validation AUC: 0.8372 <-- Best model saved
Epoch 10/20 | Train Loss: 0.4336 | Validation AUC: 0.8367
Epoch 11/20 | Train Loss: 0.4298 | Validation AUC: 0.8382 <-- Best model saved
Epoch 12/20 | Train Loss: 0.4272 | Validation AUC: 0.8379
Epoch 13/20 | Train Loss: 0.4242 | Validation AUC: 0.8369
Epoch 14/20 | Tra

In [12]:

# ============================================================
# CELL 10: RESTORE BEST MODEL AND FINAL TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    confusion_matrix,
    classification_report
)

# 1. Load the best validation checkpoint
checkpoint = torch.load(
    "/kaggle/working/dkt_model.pt",
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE)
model.eval()

print("Best model checkpoint restored successfully!")


# 2. Evaluate on the completely unseen test set
all_test_probs = []
all_test_labels = []

with torch.no_grad():

    for x, next_skill, y, mask in test_loader:

        x = x.to(DEVICE)
        next_skill = next_skill.to(DEVICE)
        y = y.to(DEVICE).float()
        mask = mask.to(DEVICE)

        # Predict logits for the next skill
        logits = step_logits(x, next_skill)

        # Convert logits into probabilities
        probabilities = torch.sigmoid(logits)

        # Collect only valid positions
        all_test_probs.append(
            probabilities[mask].cpu().numpy()
        )

        all_test_labels.append(
            y[mask].cpu().numpy()
        )


# 3. Combine predictions from all batches
y_true = np.concatenate(all_test_labels)
y_probs = np.concatenate(all_test_probs)

# Convert probabilities into binary predictions
y_pred = (y_probs >= 0.5).astype(int)


# 4. Calculate evaluation metrics
test_auc = roc_auc_score(y_true, y_probs)

test_accuracy = accuracy_score(y_true, y_pred)

test_precision = precision_score(
    y_true, y_pred, zero_division=0
)

test_recall = recall_score(
    y_true, y_pred, zero_division=0
)

test_f1 = f1_score(
    y_true, y_pred, zero_division=0
)

test_logloss = log_loss(y_true, y_probs)


# 5. Display results
print("\n" + "=" * 75)
print("FINAL TEST RESULTS")
print("=" * 75)

print(f"Test ROC-AUC : {test_auc:.4f}")
print(f"Accuracy     : {test_accuracy:.4f}")
print(f"Precision    : {test_precision:.4f}")
print(f"Recall       : {test_recall:.4f}")
print(f"F1-score     : {test_f1:.4f}")
print(f"Log Loss     : {test_logloss:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Incorrect (0)", "Correct (1)"],
        zero_division=0
    )
)

print("\nTotal test predictions:", len(y_true))
print("Test evaluation completed!")


Best model checkpoint restored successfully!

FINAL TEST RESULTS
Test ROC-AUC : 0.8267
Accuracy     : 0.7698
Precision    : 0.7768
Recall       : 0.9080
F1-score     : 0.8373
Log Loss     : 0.4593

Confusion Matrix:
[[11064 10615]
 [ 3742 36948]]

Classification Report:
               precision    recall  f1-score   support

Incorrect (0)       0.75      0.51      0.61     21679
  Correct (1)       0.78      0.91      0.84     40690

     accuracy                           0.77     62369
    macro avg       0.76      0.71      0.72     62369
 weighted avg       0.77      0.77      0.76     62369


Total test predictions: 62369
Test evaluation completed!


In [13]:
# ============================================
# CELL 11: PREDICT MASTERY WITH SKILL NAMES
# ============================================

def predict_mastery(history):
    """
    history: list of (skill_idx, correct) pairs

    Returns:
        Probability of answering each skill correctly,
        based on the student's interaction history.
    """

    model.eval()

    # Handle a student with no previous interactions
    if len(history) == 0:
        return np.full(N_SKILLS, 0.5)

    # Keep only the most recent MAX_LEN interactions
    history = history[-MAX_LEN:]

    # Separate skill indices and responses
    skills = np.array([item[0] for item in history], dtype=np.int64)
    responses = np.array([item[1] for item in history], dtype=np.int64)

    # Same encoding used during DKT training
    encoded = skills + N_SKILLS * responses

    x = torch.tensor(
        encoded,
        dtype=torch.long
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x)

        # Predictions for all skills after the final interaction
        last_logits = logits[0, -1, :]

        probabilities = torch.sigmoid(last_logits)

    return probabilities.cpu().numpy()


# --------------------------------------------
# Select one training student for demonstration
# --------------------------------------------

student_counts = train_data.groupby("user_id").size()

eligible_students = student_counts[student_counts >= 40]

if len(eligible_students) == 0:
    raise ValueError("No training student has at least 40 interactions.")

sample_user_id = eligible_students.index[0]

sample_history_df = (
    train_data[train_data["user_id"] == sample_user_id]
    .sort_values("order_id")
)

sample_history = list(
    zip(
        sample_history_df["skill_idx"].astype(int),
        sample_history_df["correct"].astype(int)
    )
)

# Predict probability for every skill
mastery_probs = predict_mastery(sample_history)


# --------------------------------------------
# Attach original skill IDs and skill names
# --------------------------------------------

mastery_results = skill_map.copy()

mastery_results["mastery_probability"] = mastery_probs
mastery_results["mastery_percentage"] = (
    mastery_results["mastery_probability"] * 100
).round(2)

# Sort from weakest to strongest predicted skills
mastery_results = mastery_results.sort_values(
    "mastery_probability",
    ascending=True
).reset_index(drop=True)


# --------------------------------------------
# Display results
# --------------------------------------------

print("=" * 70)
print("STUDENT MASTERY REPORT")
print("=" * 70)

print("Student ID:", sample_user_id)
print("Previous interactions:", len(sample_history))
print("Skills evaluated:", len(mastery_results))

print("\n10 LOWEST PREDICTED MASTERY SKILLS:")
display(
    mastery_results[
        ["skill_name", "mastery_percentage"]
    ].head(10)
)

print("\n10 HIGHEST PREDICTED MASTERY SKILLS:")
display(
    mastery_results[
        ["skill_name", "mastery_percentage"]
    ].tail(10).sort_values(
        "mastery_percentage",
        ascending=False
    )
)

# Save the complete readable report
mastery_results.to_csv(
    "/kaggle/working/student_mastery_report.csv",
    index=False
)

print("\nComplete report saved successfully!")

STUDENT MASTERY REPORT
Student ID: 14
Previous interactions: 45
Skills evaluated: 123

10 LOWEST PREDICTED MASTERY SKILLS:


,skill_name,mastery_percentage
0,Median,7.050000
1,Rotations,8.700000
2,skill_104.0,11.980000
3,Probability of Two Distinct Events,15.050000
4,Reflection,19.540001
5,Table,19.610001
6,Volume Cylinder,21.930000
7,Rate,23.360001
8,Reading a Ruler or Scale,23.629999
9,Solving Systems of Linear Equations,25.110001



10 HIGHEST PREDICTED MASTERY SKILLS:


,skill_name,mastery_percentage
122,Absolute Value,92.709999
121,Nets of 3D Figures,89.190002
120,Intercept,88.330002
119,Solving Inequalities,87.330002
118,Square Root,85.769997
117,Congruence,84.980003
116,Scatter Plot,84.480003
115,Choose an Equation from Given Information,83.959999
114,Slope,83.309998
113,Linear Equations,82.269997



Complete report saved successfully!


In [14]:

import os
from IPython.display import FileLink, display

files = [
    "/kaggle/working/dkt_model.pt",
    "/kaggle/working/skill_map.csv"
]

for path in files:
    if os.path.exists(path):
        print(f"Found: {os.path.basename(path)}")
        print(f"Size: {os.path.getsize(path) / 1024:.2f} KB")
        display(FileLink(path))
    else:
        print(f"NOT FOUND: {path}")


Found: dkt_model.pt
Size: 514.49 KB


/kaggle/working/dkt_model.pt

Found: skill_map.csv
Size: 3.74 KB


/kaggle/working/skill_map.csv